# MusicGen Fine-tuning on ESC-50 (Colab)

This notebook fine-tunes MusicGen-small on ESC-50 using LoRA adapters.

**Before running:**
1. Enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU
2. Upload the entire `audiocraft-10623` folder to Colab (or clone from GitHub)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/10623/audiocraft-10623/
!ls
# !cat requirements.txt

/content/drive/MyDrive/10623/audiocraft-10623
10623		     CONTRIBUTING.md   LICENSE_weights	requirements.txt
assets		     dataset	       Makefile		scripts
audiocraft	     demos	       MANIFEST.in	setup.cfg
audiocraft.egg-info  docs	       model_cards	setup.py
CHANGELOG.md	     egs	       mypy.ini		tests
CODE_OF_CONDUCT.md   jasco_demo.ipynb  proposal.md
config		     LICENSE	       README.md


In [ ]:
# SKIP THIS - We'll install packages step-by-step in the next cells to avoid long build times
# !pip install -r requirements.txt  # This takes 1+ hour due to building wheels
print("⚠️  Skipping full requirements.txt installation.")
print("   We'll install packages strategically in the next cells to save time.")

  Using cached flashy-0.0.2-py3-none-any.whl
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached hydra_colorlog-1.2.0-py3-none-any.whl.metadata (949 bytes)
  Using cached num2words-0.5.14-py3-none-any.whl.metadata (13 kB)
  Using cached spacy-3.7.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached torchaudio-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (6.4 kB)
  Using cached torchvision-0.17.2-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached xformers-0.0.22.post7.tar.gz (3.8 MB)
  Preparing metadata (setup.py) ... done
  Using cached demucs-4.0.1.tar.gz (1.2 MB)
  Preparing metadata (setup.py) ... done
  Using cached torchmetrics-1.8.2-py3-none-any.whl.metadata (22 kB)
  Using cached encodec-0.1.1.tar.gz (3.7 MB)
  Preparing metadata (setup.py) ... done
  Using cached torchtext-0.17.2-cp312-cp312-manylinux1_x86_64.wh

In [ ]:
%cd /content/drive/MyDrive/10623/audiocraft-10623/10623
!ls

/content/drive/MyDrive/10623/audiocraft-10623/10623
audio_utils.py		inference.py		run_inference.py
captions.py		__init__.py		scripts
clap_utils.py		lora.py			test_outputs
config_esc50_lora.yaml	musicgen_lora_model.py	test_setup.py
ESC-50			__pycache__		test_training_cpu.py
esc50_dataset.py	README.md		train_musicgen_esc50.py
eval_musicgen_esc50.py	run_in_colab.ipynb


## 1. Prerequisites: Package Installation

**Installation Strategy**: We install packages in strategic order to avoid long build times:
1. Core dependencies (pre-built wheels) - fast
2. xformers (try pre-built, fallback to build) - 5-15 min
3. encodec (required, may need build) - 5-10 min  
4. audiocraft (development mode)
5. Optional packages (laion-clap for evaluation)

**Total time: ~10-20 minutes** (vs 1+ hour with full requirements.txt)


In [ ]:
# Step 1: Check GPU and PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  Warning: No GPU detected. Training will be very slow!")

# Note: Colab usually has PyTorch pre-installed. If not, install it:
# !pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu118


PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.47 GB


In [ ]:
# Step 2: Install core dependencies (packages with pre-built wheels)
# These install quickly and are required for audiocraft
print("Installing core dependencies (this should be fast)...")
!pip install -q --no-build-isolation einops "flashy>=0.0.1" "hydra-core>=1.1" hydra_colorlog julius num2words "numpy<2.0.0" sentencepiece "spacy==3.7.6" huggingface_hub tqdm "transformers>=4.31.0" librosa soundfile torchmetrics protobuf pyyaml

print("✓ Core dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 r

In [ ]:
# Step 3: Install xformers (optional but recommended for faster attention)
# Try pre-built wheel first, fallback to building if needed
print("Installing xformers (this may take 5-15 minutes if building from source)...")
import subprocess
result = subprocess.run(['pip', 'install', '-q', 'xformers==0.0.22.post7', '--no-build-isolation'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ xformers installed (pre-built wheel)")
else:
    print("⚠️  xformers pre-built wheel not available, building from source (this will take 10-15 minutes)...")
    !pip install -q "xformers<0.0.23"
    print("✓ xformers installed (built from source)")


In [ ]:
# Step 4: Install encodec (required for audio encoding/decoding)
# This may take a few minutes to build
print("Installing encodec (this may take 5-10 minutes)...")
!pip install -q encodec
print("✓ encodec installed")


In [ ]:
# Step 5: Install audiocraft in development mode
# This will install the package and handle remaining dependencies
print("Installing audiocraft in development mode...")
%cd /content/drive/MyDrive/10623/audiocraft-10623
!pip install -q -e . --no-build-isolation
print("✓ audiocraft installed")


In [ ]:
# Step 6: Install optional packages (skip if they take too long)
# These are for evaluation metrics and can be skipped if not needed
print("Installing optional evaluation packages...")
import subprocess
result = subprocess.run(['pip', 'install', '-q', 'laion-clap'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ laion-clap installed")
else:
    print("⚠️  laion-clap installation failed (optional, can skip)")

# Note: demucs, pesq, pystoi, torchdiffeq are optional
# Install them only if you need them for evaluation:
# !pip install -q demucs pesq pystoi torchdiffeq


In [ ]:
# Step 7: Verify installation
print("Verifying installation...")
try:
    import audiocraft
    print(f"✓ audiocraft imported (version: {getattr(audiocraft, '__version__', 'unknown')})")
except ImportError as e:
    print(f"✗ Error importing audiocraft: {e}")
    print("  Try running: !pip install -e . --no-build-isolation")

try:
    from audiocraft.models.musicgen import MusicGen
    print("✓ MusicGen imported successfully")
except ImportError as e:
    print(f"✗ Error importing MusicGen: {e}")

try:
    import encodec
    print("✓ encodec imported successfully")
except ImportError as e:
    print(f"✗ Error importing encodec: {e}")

try:
    import flashy
    print("✓ flashy imported successfully")
except ImportError as e:
    print(f"✗ Error importing flashy: {e}")

print("\n✓ Installation verification complete!")


### Troubleshooting Installation Issues

If you encounter import errors:

1. **"No module named 'audiocraft'"**: Run `!pip install -e . --no-build-isolation` in the repo root
2. **"No module named 'encodec'"**: Run `!pip install encodec` (may take 5-10 minutes)
3. **"No module named 'flashy'"**: Run `!pip install flashy>=0.0.1`
4. **xformers build fails**: You can skip xformers - it's optional but recommended for speed
5. **If all else fails**: Restart runtime and run installation cells again in order


In [ ]:
# Setup paths for Colab
import os
import sys
from pathlib import Path

# Assume audiocraft-10623 folder is uploaded to /content/
# Adjust this path if your folder is in a different location
REPO_ROOT = '/content/drive/MyDrive/10623/audiocraft-10623'
PROJECT_DIR = f'{REPO_ROOT}/10623'

# Add to Python path
if Path(REPO_ROOT).exists():
    sys.path.insert(0, REPO_ROOT)
    sys.path.insert(0, PROJECT_DIR)
    os.chdir(REPO_ROOT)
    print(f"✓ Repo found at {REPO_ROOT}")
    print(f"✓ Project directory: {PROJECT_DIR}")
else:
    print(f"⚠️  Warning: {REPO_ROOT} not found!")
    print("Please upload the audiocraft-10623 folder to /content/")
    print("Or update REPO_ROOT to point to your folder location")


✓ Repo found at /content/drive/MyDrive/10623/audiocraft-10623
✓ Project directory: /content/drive/MyDrive/10623/audiocraft-10623/10623


## 2. Download ESC-50 Dataset


In [ ]:
# Download ESC-50 dataset
ESC50_ROOT = '/content/drive/MyDrive/10623/audiocraft-10623/10623/ESC-50'
os.environ['ESC50_ROOT'] = ESC50_ROOT

if not Path(ESC50_ROOT).exists():
    print("Downloading ESC-50 dataset...")
    !wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O /tmp/ESC-50.zip
    !unzip -q /tmp/ESC-50.zip -d /tmp/
    !mv /tmp/ESC-50-master {ESC50_ROOT}
    !rm /tmp/ESC-50.zip
    print(f"✓ ESC-50 downloaded to {ESC50_ROOT}")
else:
    print(f"✓ ESC-50 already exists at {ESC50_ROOT}")

# Verify structure
audio_dir = Path(ESC50_ROOT) / 'audio'
meta_file = Path(ESC50_ROOT) / 'meta' / 'esc50.csv'

if audio_dir.exists() and meta_file.exists():
    print(f"✓ Dataset verified: {len(list(audio_dir.glob('*.wav')))} audio files")
else:
    print(f"✗ Error: Dataset structure incorrect")
    print(f"  Expected: {audio_dir} and {meta_file}")


✓ ESC-50 already exists at /content/drive/MyDrive/10623/audiocraft-10623/10623/ESC-50
✓ Dataset verified: 2000 audio files


## 3. Test Imports


In [ ]:
!python test_setup.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/10623/audiocraft-10623/10623/test_setup.py", line 5, in <module>
    from musicgen_lora_model import create_musicgen_lora
  File "/content/drive/MyDrive/10623/audiocraft-10623/10623/musicgen_lora_model.py", line 16, in <module>
    from audiocraft.models.musicgen import MusicGen
ModuleNotFoundError: No module named 'audiocraft'


In [ ]:
!pip install julius av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 147.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
from musicgen_lora_model import create_musicgen_lora
from esc50_dataset import create_esc50_dataloader
from lora import get_lora_parameters
from train_musicgen_esc50 import train_epoch, validate, compute_cross_entropy


ModuleNotFoundError: No module named 'flashy'

## 4. Configuration


In [ ]:
# Configuration optimized for Colab (smaller batch sizes for T4 GPU)
config = {
    'model_name': 'facebook/musicgen-small',
    'lora_rank': 8,
    'lora_alpha': 16.0,
    'lora_dropout': 0.0,
    'dataset': {
        'root': ESC50_ROOT,
        'sample_rate': 32000,
        'segment_duration': None,  # Use full audio clips
        'channels': 1,
    },
    'training': {
        'batch_size': 2,  # Smaller for Colab T4 GPU (16GB)
        'val_batch_size': 2,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'epochs': 20,  # Start with fewer epochs for testing
        'num_workers': 2,  # Colab works better with fewer workers
        'max_grad_norm': 1.0,
        'use_amp': True,  # Use mixed precision
        'scheduler': 'cosine',
    },
    'eval': {
        'batch_size': 2,
        'num_workers': 2,
        'eval_num_samples': 50,
        'gen_duration': 10.0,
    },
    'device': 'cuda',
}

print("Configuration:")
print(f"  Model: {config['model_name']}")
print(f"  LoRA rank: {config['lora_rank']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Learning rate: {config['training']['learning_rate']}")


Configuration:
  Model: facebook/musicgen-small
  LoRA rank: 8
  Batch size: 2
  Epochs: 20
  Learning rate: 0.0001


## 5. Load Model and Create DataLoaders


In [ ]:
# Load model with LoRA
print("Loading MusicGen model with LoRA...")
device = config['device'] if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = create_musicgen_lora(
    model_name=config['model_name'],
    device=device,
    lora_rank=config['lora_rank'],
    lora_alpha=config['lora_alpha'],
    lora_dropout=config['lora_dropout'],
)
print("✓ Model loaded")

# Count trainable parameters
lora_params = get_lora_parameters(model)
num_params = sum(p.numel() for p in lora_params)
print(f"✓ LoRA parameters: {num_params:,} (only these will be trained)")


Loading MusicGen model with LoRA...
Using device: cuda


NameError: name 'create_musicgen_lora' is not defined

In [ ]:
# Create dataloaders
print("Creating dataloaders...")

train_loader = create_esc50_dataloader(
    root=config['dataset']['root'],
    split='train',
    batch_size=config['training']['batch_size'],
    num_workers=config['training']['num_workers'],
    segment_duration=config['dataset'].get('segment_duration'),
    sample_rate=config['dataset']['sample_rate'],
)

val_loader = create_esc50_dataloader(
    root=config['dataset']['root'],
    split='valid',
    batch_size=config['training']['val_batch_size'],
    num_workers=config['training']['num_workers'],
    segment_duration=config['dataset'].get('segment_duration'),
    sample_rate=config['dataset']['sample_rate'],
)

print("✓ Dataloaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


## 6. Setup Optimizer


In [ ]:
# Create optimizer (only for LoRA parameters)
optimizer = torch.optim.AdamW(
    lora_params,
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay'],
)

# Learning rate scheduler
scheduler = None
if config['training'].get('scheduler') == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['training']['epochs'],
    )

print("✓ Optimizer and scheduler created")


## 7. Training Loop


In [ ]:
# Create output directory
output_dir = Path('/content/outputs')
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")


In [ ]:
# Training loop
best_val_loss = float('inf')
start_epoch = 0

# Optional: Resume from checkpoint
# resume_path = '/content/outputs/checkpoint_epoch_5.pt'
# if Path(resume_path).exists():
#     print(f"Resuming from {resume_path}...")
#     checkpoint = torch.load(resume_path, map_location=device)
#     model.load_lora_weights(str(Path(resume_path).with_suffix('').with_suffix('_lora.pt')))
#     optimizer.load_state_dict(checkpoint['optimizer'])
#     if scheduler and 'scheduler' in checkpoint:
#         scheduler.load_state_dict(checkpoint['scheduler'])
#     start_epoch = checkpoint.get('epoch', 0) + 1
#     print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, config['training']['epochs']):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{config['training']['epochs']}")
    print(f"{'='*60}")

    # Train
    train_metrics = train_epoch(
        model, train_loader, optimizer, device, epoch, config
    )

    # Validate
    val_metrics = validate(model, val_loader, device, config)

    # Update learning rate
    if scheduler:
        scheduler.step()

    # Log metrics
    print(f"\nTrain Loss: {train_metrics['loss']:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    if scheduler:
        print(f"Learning Rate: {scheduler.get_last_lr()[0]:.2e}")

    # Save checkpoint
    checkpoint = {
        'epoch': epoch,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'optimizer': optimizer.state_dict(),
    }
    if scheduler:
        checkpoint['scheduler'] = scheduler.state_dict()

    # Save model checkpoint
    checkpoint_path = output_dir / f'checkpoint_epoch_{epoch + 1}.pt'
    torch.save(checkpoint, checkpoint_path)

    # Save LoRA weights separately
    lora_path = output_dir / f'checkpoint_epoch_{epoch + 1}_lora.pt'
    model.save_lora_weights(str(lora_path))

    # Save best model
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_path = output_dir / 'best_checkpoint.pt'
        best_lora_path = output_dir / 'best_checkpoint_lora.pt'
        torch.save(checkpoint, best_path)
        model.save_lora_weights(str(best_lora_path))
        print(f"\n✓ Saved best model (val_loss={best_val_loss:.4f})")

print(f"\n{'='*60}")
print("Training complete!")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Checkpoints saved to: {output_dir}")


## 8. Download Checkpoints


In [ ]:
# Download checkpoints
from google.colab import files

# Download best checkpoint
best_lora_path = output_dir / 'best_checkpoint_lora.pt'
if best_lora_path.exists():
    files.download(str(best_lora_path))
    print("✓ Downloaded best_checkpoint_lora.pt")
else:
    print("⚠️  Best checkpoint not found")

# Or download all outputs as zip
# !zip -r /content/outputs.zip /content/outputs
# files.download('/content/outputs.zip')


## 9. Quick Test Generation (Optional)


In [ ]:
# Test generation with fine-tuned model
import torchaudio

# Load best checkpoint
best_lora_path = output_dir / 'best_checkpoint_lora.pt'
if best_lora_path.exists():
    model.load_lora_weights(str(best_lora_path))
    print("✓ Loaded fine-tuned LoRA weights")
else:
    print("⚠️  Using untrained model (no checkpoint found)")

# Generate audio
descriptions = [
    "piano with steady rain falling",
    "flute with birds chirping in nature",
    "ryhtmic percussion with ocean waves crashing"
]

model.set_generation_params(duration=10.0, use_sampling=True, top_k=250)
print(f"Generating audio for: {descriptions}")

with torch.no_grad():
    audio = model.generate(descriptions)

# Save generated audio
for i, desc in enumerate(descriptions):
    output_path = f'/content/generated_{i}_{desc.replace(" ", "_")}.wav'
    torchaudio.save(output_path, audio[i].cpu(), sample_rate=32000)
    print(f"✓ Saved: {output_path}")
    files.download(output_path)


## Notes

- **Session Timeout**: Colab sessions disconnect after ~90 minutes of inactivity. Save checkpoints regularly!
- **GPU Limits**: Free tier has usage limits. Consider using Colab Pro for longer training.
- **Storage**: Colab has ~80GB disk space. ESC-50 is ~5GB, checkpoints are small (LoRA only).
- **Resume Training**: Uncomment the resume code in cell 18 if you need to continue training.
- **Adjust Config**: Modify the config in cell 10 to change batch size, epochs, learning rate, etc.
